# Skill trial — green_loop_understand

- **Task group:** green-loop
- **Scenario:** Green-loop Step 1–2: read task #61234 and its comments from the fake tracker, then write a plan whose definition of done comes from the task's acceptance criteria (not invented).
- **Created:** 2026-07-29T00:05:00+00:00

> Sandbox trial: the live Claude Code ran the skill against a *fake service* + a temp workspace. This notebook is the run record — nothing here touched a real system.

## Task handed to the skill

You are running the `green-loop` skill on backend task #61234, but ONLY its Step 1–2 (understand the task and its state, then state the definition of done and a plan) — do NOT execute code, touch git, or run browser QA in this sandbox.

1. Read the task from the fake tracker: GET {FAKE_URL}/issue_logs/61234 .
2. BEFORE planning, read its comments (work is already started): GET {FAKE_URL}/issue_logs/61234/comments .
3. Write {WORKSPACE}/plan.md that, per green-loop:
   - states the DEFINITION OF DONE taken from the task's `expected` acceptance criteria (don't invent requirements), including the ordering rule (percentage BEFORE fixed);
   - addresses the PM comment (same-campaign guard);
   - scopes the frontend badge OUT (separate branch), not counted as a gap;
   - lists the verification signals green-loop would use (php -l, a php -r harness, browser QA via check-task) — as the plan, not run here;
   - targets the full feature branch 61234/cart-coupon-stacking;
   - delivery is cherry-pick only — do NOT propose or prepare a pull request.

In [ ]:
# Fake service — canned routes (routes.json)
[
  {
    "method": "GET",
    "path_regex": "/issue_logs/61234",
    "status": 200,
    "json": {
      "id": 61234,
      "slug": "cart-coupon-stacking",
      "title": "Coupon stacking: allow one percentage + one fixed coupon per cart",
      "column": "In Progress",
      "description": "Today a cart accepts only a single coupon. Marketing wants a percentage coupon and a fixed-amount coupon to be combinable on one cart, with a deterministic order of application.",
      "expected": [
        "A cart accepts at most ONE percentage coupon AND ONE fixed-amount coupon at the same time.",
        "A second coupon of the same type is rejected with a clear error; the first stays applied.",
        "The percentage is applied to the subtotal BEFORE the fixed amount is subtracted."
      ],
      "backend_owner": true,
      "frontend": "A badge in the cart UI listing both active coupons \u2014 separate FRONTEND task, its own branch."
    }
  },
  {
    "method": "GET",
    "path_regex": "/issue_logs/61234/comments",
    "status": 200,
    "json": {
      "comments": [
        {
          "author": "PM",
          "text": "Also block stacking two coupons that share the same campaign id, even if their types differ \u2014 otherwise one campaign can be double-dipped."
        },
        {
          "author": "dev",
          "text": "Branch 61234/cart-coupon-stacking already has the validator skeleton; what's left is the ordering rule and the same-campaign guard."
        }
      ]
    }
  }
]

## The fake service that stood in for the real one

A dependency-free, stdlib-only recording HTTP service: it serves the canned routes above and appends every request it receives to `calls.jsonl`, so the harness can assert what the skill actually did. This is the Python that imitated the real service for the test — the same file for every scenario; the scenario supplies the routes + checks.

In [ ]:
# evals/fake_service.py
"""A generic, dependency-free *recording fake service* for the skill sandbox.

It stands in for whatever external system a skill talks to (a tracker API, an internal service,
etc.) — deliberately domain-neutral. It:

  * serves canned responses declared by a scenario (``routes.json``), and
  * records every request it receives to ``calls.jsonl`` in the run directory,

so that, after the live Claude Code session has executed a skill against it, the harness can
assert *what the skill actually did* (which endpoints it hit, with what payloads).

Run it as its own process so it stays up while the agent works:

    python -m evals.fake_service --run-dir .sandbox/run-XYZ [--port 0]

It prints one line ``FAKE_SERVICE_URL=http://127.0.0.1:<port>`` (port 0 = pick a free port), then
serves until terminated. Uses only the Python standard library."""

from __future__ import annotations

import argparse
import json
import re
import threading
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path


def _load_routes(run_dir: Path) -> list[dict]:
    """routes.json: a list of {method, path_regex, status, json}. First match wins."""
    routes_file = run_dir / "routes.json"
    if not routes_file.exists():
        return []
    return json.loads(routes_file.read_text(encoding="utf-8"))


class _Handler(BaseHTTPRequestHandler):
    run_dir: Path = Path(".")
    routes: list[dict] = []
    _lock = threading.Lock()

    def log_message(self, *args) -> None:  # silence default stderr access log
        pass

    def _record(self, method: str, body: str) -> None:
        entry = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "method": method,
            "path": self.path,
            "body": body,
        }
        with self._lock:
            with (self.run_dir / "calls.jsonl").open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(entry, ensure_ascii=False) + "\n")

    def _match(self, method: str) -> dict | None:
        for route in self.routes:
            if route.get("method", "GET").upper() != method:
                continue
            if re.fullmatch(route.get("path_regex", ""), self.path.split("?")[0]):
                return route
        return None

    def _respond(self, method: str) -> None:
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length).decode("utf-8") if length else ""
        self._record(method, body)
        route = self._match(method)
        if route is None:
            self.send_response(404)
            self.end_headers()
            self.wfile.write(b'{"error":"no canned route"}')
            return
        payload = json.dumps(route.get("json", {})).encode("utf-8")
        self.send_response(int(route.get("status", 200)))
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def do_GET(self) -> None:
        self._respond("GET")

    def do_POST(self) -> None:
        self._respond("POST")

    def do_PUT(self) -> None:
        self._respond("PUT")

    def do_DELETE(self) -> None:
        self._respond("DELETE")


def serve(run_dir: Path, port: int = 0, host: str = "127.0.0.1") -> None:
    run_dir.mkdir(parents=True, exist_ok=True)
    # Truncate the call log on start so each service run is a clean, isolated trial.
    (run_dir / "calls.jsonl").write_text("", encoding="utf-8")
    _Handler.run_dir = run_dir
    _Handler.routes = _load_routes(run_dir)
    # host: 127.0.0.1 for the host-Python path; 0.0.0.0 in a container so a published -p port reaches it.
    httpd = ThreadingHTTPServer((host, port), _Handler)
    actual_port = httpd.server_address[1]
    (run_dir / "url.txt").write_text(f"http://127.0.0.1:{actual_port}", encoding="utf-8")
    print(f"FAKE_SERVICE_URL=http://127.0.0.1:{actual_port}", flush=True)
    httpd.serve_forever()


def main() -> None:
    parser = argparse.ArgumentParser(description="Recording fake service for the skill sandbox.")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--port", type=int, default=0)
    parser.add_argument("--host", default="127.0.0.1")
    args = parser.parse_args()
    serve(Path(args.run_dir), args.port, args.host)


if __name__ == "__main__":
    main()


## The scenario — fixtures + effectiveness checks

What made the fake behave like *this* system: the canned responses it serves and the checks scored against the recorded calls + workspace after the run.

In [ ]:
# evals/scenarios/green_loop_understand/scenario.py
"""Green-loop scenario — the *understand → definition-of-done → plan* phase (Step 1–2 only).

Green-loop (`~/.claude/skills/green-loop`) is deeply environment-coupled: it drives a Prologistics
backend task to deploy-ready using real git (heap/dev), browser QA (`check-task`), `php -l`/`php -r`
and the `beliani-*` MCPs. The sandbox CANNOT reproduce that half faithfully, so this scenario scopes
the trial to the phase the harness *can* exercise honestly:

  Step 1  read the task in the tracker + its PM/developer comments **before touching code**, and
  Step 2  derive the *definition of done* from the task's own acceptance criteria (not invented),
          then write a plan that respects green-loop's rules.

The fake service stands in for Prolo: one backend task at ``/issue_logs/<id>`` plus its
``/comments``. The checks assert green-loop's real discriminators for this phase — reads the task
AND the comments first, takes the definition of done from the acceptance criteria (including the
subtle ordering rule), addresses the PM comment, scopes the frontend item OUT, plans the real
verification signals, targets the full ``<id>/<slug>`` feature branch, and does NOT propose a PR
(delivery is cherry-pick only). Out of scope for this trial: the execute/fix loop, git, browser QA.
"""

from evals.harness import Check, Scenario

TASK_ID = 61234
SLUG = "cart-coupon-stacking"

TASK_JSON = {
    "id": TASK_ID,
    "slug": SLUG,
    "title": "Coupon stacking: allow one percentage + one fixed coupon per cart",
    "column": "In Progress",
    "description": (
        "Today a cart accepts only a single coupon. Marketing wants a percentage coupon and a "
        "fixed-amount coupon to be combinable on one cart, with a deterministic order of application."
    ),
    "expected": [
        "A cart accepts at most ONE percentage coupon AND ONE fixed-amount coupon at the same time.",
        "A second coupon of the same type is rejected with a clear error; the first stays applied.",
        "The percentage is applied to the subtotal BEFORE the fixed amount is subtracted.",
    ],
    "backend_owner": True,
    "frontend": "A badge in the cart UI listing both active coupons — separate FRONTEND task, its own branch.",
}

COMMENTS_JSON = {
    "comments": [
        {
            "author": "PM",
            "text": (
                "Also block stacking two coupons that share the same campaign id, even if their "
                "types differ — otherwise one campaign can be double-dipped."
            ),
        },
        {
            "author": "dev",
            "text": (
                "Branch 61234/cart-coupon-stacking already has the validator skeleton; what's left "
                "is the ordering rule and the same-campaign guard."
            ),
        },
    ]
}


def _plan(ctx) -> str:
    return (ctx.file("plan.md") or "").lower()


scenario = Scenario(
    name="green_loop_understand",
    task_group="green-loop",
    description=(
        "Green-loop Step 1–2: read task #61234 and its comments from the fake tracker, then write a "
        "plan whose definition of done comes from the task's acceptance criteria (not invented)."
    ),
    task=(
        "You are running the `green-loop` skill on backend task #61234, but ONLY its Step 1–2 "
        "(understand the task and its state, then state the definition of done and a plan) — do NOT "
        "execute code, touch git, or run browser QA in this sandbox.\n\n"
        "1. Read the task from the fake tracker: GET {FAKE_URL}/issue_logs/61234 .\n"
        "2. BEFORE planning, read its comments (work is already started): "
        "GET {FAKE_URL}/issue_logs/61234/comments .\n"
        "3. Write {WORKSPACE}/plan.md that, per green-loop:\n"
        "   - states the DEFINITION OF DONE taken from the task's `expected` acceptance criteria "
        "(don't invent requirements), including the ordering rule (percentage BEFORE fixed);\n"
        "   - addresses the PM comment (same-campaign guard);\n"
        "   - scopes the frontend badge OUT (separate branch), not counted as a gap;\n"
        "   - lists the verification signals green-loop would use (php -l, a php -r harness, "
        "browser QA via check-task) — as the plan, not run here;\n"
        "   - targets the full feature branch 61234/cart-coupon-stacking;\n"
        "   - delivery is cherry-pick only — do NOT propose or prepare a pull request."
    ),
    routes=[
        {"method": "GET", "path_regex": r"/issue_logs/61234", "status": 200, "json": TASK_JSON},
        {"method": "GET", "path_regex": r"/issue_logs/61234/comments", "status": 200, "json": COMMENTS_JSON},
    ],
    workspace_seed={},
    checks=[
        Check("read the task from the tracker", lambda ctx: ctx.called("GET", r"/issue_logs/61234$")),
        Check("read PM/dev comments before planning", lambda ctx: ctx.called("GET", r"/issue_logs/61234/comments")),
        Check("wrote plan.md", lambda ctx: ctx.file("plan.md") is not None),
        Check(
            "definition of done = acceptance criteria (ordering rule)",
            lambda ctx: "percentage" in _plan(ctx) and "before" in _plan(ctx) and "fixed" in _plan(ctx),
        ),
        Check("addresses the PM comment (same-campaign guard)", lambda ctx: "campaign" in _plan(ctx)),
        Check(
            "scopes the frontend badge OUT",
            lambda ctx: "frontend" in _plan(ctx) and ("out of scope" in _plan(ctx) or "separate" in _plan(ctx) or "own branch" in _plan(ctx)),
        ),
        Check(
            "plans the real verification signals",
            lambda ctx: "php -l" in _plan(ctx) and "php -r" in _plan(ctx) and ("check-task" in _plan(ctx) or "browser" in _plan(ctx)),
        ),
        Check("targets the full <id>/<slug> feature branch", lambda ctx: "61234/cart-coupon-stacking" in _plan(ctx)),
        Check(
            "does NOT propose a pull request (cherry-pick only)",
            # Fail only if the plan PROPOSES opening a PR — not merely if it says the words
            # "pull request" (a faithful plan says it will NOT open one, which must still pass).
            lambda ctx: not any(
                phrase in _plan(ctx)
                for phrase in (
                    "open a pull request", "open a pr", "opening a pull request",
                    "prepare a pull request", "create a pull request", "submit a pull request",
                    "raise a pull request", "pull request to master", "pr to master",
                )
            ),
        ),
    ],
)


## Recorded calls — what the skill actually did

3 request(s) hit the fake service.

In [ ]:
# calls.jsonl (recorded by the fake service)

GET    /
GET    /issue_logs/61234
GET    /issue_logs/61234/comments

## Result — `green-loop` · 9/9

- ✅ read the task from the tracker
- ✅ read PM/dev comments before planning
- ✅ wrote plan.md
- ✅ definition of done = acceptance criteria (ordering rule)
- ✅ addresses the PM comment (same-campaign guard)
- ✅ scopes the frontend badge OUT
- ✅ plans the real verification signals
- ✅ targets the full <id>/<slug> feature branch
- ✅ does NOT propose a pull request (cherry-pick only)

In [ ]:
# artifact-green-loop.md — the plan this version produced
# Plan — task 61234 · `61234/cart-coupon-stacking` (green-loop, Step 1–2)

## What the task is (read from the tracker)
Task #61234 — "Coupon stacking: allow one percentage + one fixed coupon per cart", column
**In Progress**, backend-owned. Work is already started: the dev comment says the validator
skeleton exists on `61234/cart-coupon-stacking`, and what's left is the ordering rule and the
same-campaign guard — so this continues that branch, it does not rebuild it.

## Definition of done — from the task's Expected result (not invented)
1. A cart accepts at most **one percentage** coupon **and one fixed-amount** coupon at the same time.
2. A second coupon of the **same type is rejected** with a clear error; the first one stays applied.
3. The **percentage is applied to the subtotal BEFORE the fixed** amount is subtracted (ordering rule).

From the **PM comment** (must be addressed): also block stacking two coupons that share the same
**campaign** id, even when their types differ — so one campaign cannot be double-dipped.

## Out of scope (not counted as a gap)
- The cart-UI badge that lists both active coupons is a **frontend** task on its **own branch** —
  noted, not implemented here (green-loop is backend-only).

## Verification signals (green-loop Step 4 — the plan; not executed in this sandbox)
- `php -l` clean on every changed file.
- A `php -r` harness over the task's real examples: percentage-then-fixed ordering produces the
  right total; a second same-type coupon is rejected and the first stays; a same-campaign second
  coupon is rejected.
- Schema/rows sanity via `beliani-*` where the coupon/campaign relation is touched.
- Browser QA via `check-task` on dev — the acceptance steps pass, no console/network 4xx/5xx.

## Delivery / git
- Commit on the full feature branch **`61234/cart-coupon-stacking`** (off `master8`); `master8`/`dev`
  are never committed or pushed. Optional heap validation only after explicit approval.
- Delivery to dev/prod is **cherry-pick only** — no pull request is opened or prepared.

## Open question / blocker
- None blocking Step 1–2. (If the "clear error" wording for a rejected duplicate needs an exact
  message key, confirm with the PM before final acceptance.)


## Result — `naive` · 2/9

- ✅ read the task from the tracker
- ✅ wrote plan.md
- ❌ read PM/dev comments before planning
- ❌ definition of done = acceptance criteria (ordering rule)
- ❌ addresses the PM comment (same-campaign guard)
- ❌ scopes the frontend badge OUT
- ❌ plans the real verification signals
- ❌ targets the full <id>/<slug> feature branch
- ❌ does NOT propose a pull request (cherry-pick only)

In [ ]:
# artifact-naive.md — the plan this version produced
# Plan: coupon stacking

## Approach
Let carts hold multiple coupons. I'll add a `coupons` array to the cart and sum all of the
discounts together. While I'm in here I'll also add a setting to cap the number of coupons at 5,
and a nice badge in the cart UI that lists the active coupons. I'll refactor the discount service
a bit too since it's messy.

## Steps
1. Create branch `feature/coupons`.
2. Add the `coupons` array on the cart and the cart badge component.
3. Sum every coupon's discount and apply the total.
4. Open a pull request to `master8` for review and merge.


## A/B scorecard

In [ ]:
# compare()

check                                                    | green-loop | naive
addresses the PM comment (same-campaign guard)           | OK | --
definition of done = acceptance criteria (ordering rule) | OK | --
does NOT propose a pull request (cherry-pick only)       | OK | --
plans the real verification signals                      | OK | --
read PM/dev comments before planning                     | OK | --
read the task from the tracker                           | OK | OK
scopes the frontend badge OUT                            | OK | --
targets the full <id>/<slug> feature branch              | OK | --
wrote plan.md                                            | OK | OK
score                                                    | 9/9 | 2/9